In [ ]:
# Import necessary libraries
from pathlib import Path
import re
import numpy as np
import pandas as pd

In [ ]:
# Config 
INPUT = Path(r"data evaluation\Questionnaire - Type 2 Diabetes Dialogues (Responses) - Form responses 1.csv")
OUT_DIR = Path("human_eval_outputs"); OUT_DIR.mkdir(exist_ok=True, parents=True)

# Constants
DIMENSIONS = ["Fluency", "Naturalness", "Clarity", "Structure"]
COL_PATTERN = re.compile(r"^(Fluency|Naturalness|Clarity|Structure)\s*\(Conv\s*No\.?\s*(\d+)\s*\)$", re.IGNORECASE)
RATER_COL = "Rater ID"

In [ ]:
# Helper functions
def convert_score(x):
    """
    Convert Likert to numeric 1–5. Works for '4', '4 (Good)', 'Good', etc.
    Returns np.nan if cannot parse or out of range.

    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x) if 1 <= float(x) <= 5 else np.nan
    s = str(x).strip()
    m = re.search(r"\b([1-5])\b", s)
    if m:
        return float(m.group(1))
    s_low = s.lower()
    mapping = {
        "very poor": 1,
        "poor": 2,           # will not override "very poor" due to order above
        "acceptable": 3,
        "good": 4,
        "excellent": 5,
    }
    for k, v in mapping.items():
        if k in s_low:
            return float(v)
    return np.nan

def extract_cols(df):
    """
    Return list of rating columns and parsed metadata (dimension, conversation_id).
    
    """
    parsed = []
    for col in df.columns:
        m = COL_PATTERN.match(col.strip())
        if m:
            dim = m.group(1).title()
            conv = int(m.group(2))
            parsed.append((col, dim, conv))
    return parsed

In [5]:
# Load data
df_raw = pd.read_csv(INPUT, encoding="utf-8")

# Ensure a rater identifier
if RATER_COL not in df_raw.columns:
    # Fallback: create one from row index
    df_raw[RATER_COL] = [f"R{i+1}" for i in range(len(df_raw))]

In [ ]:
# Format
parsed_cols = extract_cols(df_raw)
records = []
for _, row in df_raw.iterrows():
    rater_id = str(row[RATER_COL]).strip()
    for col, dim, conv in parsed_cols:
        score = convert_score(row[col])
        records.append({
            "rater_id": rater_id,
            "dimension": dim,
            "conversation_id": int(conv),
            "score": score
        })

long_df = pd.DataFrame(records)

# Keep only the four target dimensions in a consistent order
long_df["dimension"] = pd.Categorical(long_df["dimension"], categories=DIMENSIONS, ordered=True)

# Save tidy data
long_path = OUT_DIR / "human_eval_long.csv"
long_df.to_csv(long_path, index=False)
print(f"Saved tidy ratings → {long_path}")

## SUMMARY STATS
# Per-dimension across all conversations and raters
summary_dim = long_df.groupby("dimension", as_index=False).agg(
    N=("score", "count"),
    mean=("score", "mean"),
    sd=("score", "std")
)
summary_dim = summary_dim.sort_values("dimension")
summary_path = OUT_DIR / "human_eval_summary_by_dimension.csv"
summary_dim.to_csv(summary_path, index=False)
print(f"Saved per-dimension summary → {summary_path}")

# Per conversation × dimension
summary_conv_dim = long_df.groupby(["conversation_id", "dimension"], as_index=False).agg(
    N=("score", "count"),
    mean=("score", "mean"),
    sd=("score", "std")
).sort_values(["conversation_id", "dimension"])
summary_conv_path = OUT_DIR / "human_eval_summary_by_conversation_dimension.csv"
summary_conv_dim.to_csv(summary_conv_path, index=False)
print(f"Saved per conversation × dimension summary → {summary_conv_path}")

# Krippendorff's alpha (ordinal) per dimension
try:
    import krippendorff
except ImportError as e:
    raise ImportError(
        "The 'krippendorff' package is required. Install it via:\n"
        "  pip install krippendorff"
    ) from e

alphas = {}
for dim in DIMENSIONS:
    sub = long_df[long_df["dimension"] == dim].copy()
    # Pivot to matrix: rows = raters, cols = conversations
    mat = sub.pivot_table(index="rater_id", columns="conversation_id", values="score", aggfunc="first")
    # Convert to numpy with np.nan where missing
    data = mat.to_numpy()
    # krippendorff.alpha expects shape (n_raters, n_units)
    alpha = krippendorff.alpha(
        reliability_data=data,
        level_of_measurement='ordinal',
        value_domain=[1, 2, 3, 4, 5]
    )
    alphas[dim] = float(alpha)

alpha_df = pd.DataFrame([
    {"dimension": dim, "krippendorff_alpha_ordinal": val}
    for dim, val in alphas.items()
]).sort_values("dimension")
alpha_path = OUT_DIR / "human_eval_krippendorff_alpha.csv"
alpha_df.to_csv(alpha_path, index=False)
print(f"Saved Krippendorff’s alpha per dimension → {alpha_path}")

# Print Review
print("\n=== Per-dimension means ± SD ===")
for _, r in summary_dim.iterrows():
    print(f"{r['dimension']}: {r['mean']:.3f} ± {r['sd']:.3f} (N={int(r['N'])})")

print("\n=== Krippendorff’s α (ordinal) per dimension ===")
for _, r in alpha_df.iterrows():
    print(f"{r['dimension']}: {r['krippendorff_alpha_ordinal']:.3f}")